# CPV301 AutoDrive — D1 governed Colab training

This notebook restores only the approved **train/val** bundle, checks its embedded commit and stem hashes, synchronizes `detector/uv.lock`, and runs `m1.cli`. It never uploads, exports, or reads the sealed test. Logs are persisted continuously to private Drive as both `colab_train.log` and `DB_logger.db`; the SQLite table matches the retained VGG16 notebook (`timestamp`, `level`, `logger`, `message`).

Keep `SMOKE = True` for the required one-epoch smoke. Change it only after that smoke result is downloaded and independently reviewed.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import hashlib
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/CPV391/M1_D1_Colab')
DRIVE_BUNDLE = DRIVE_ROOT / 'm1_colab_train_val_backend_r1.tar.gz'
SIDECAR = DRIVE_ROOT / 'm1_colab_train_val_backend_r1.tar.gz.manifest.json'
EXPECTED_BUNDLE_BYTES = 766226800
EXPECTED_BUNDLE_SHA256 = '7ea344cb1e2c166786d0552f215993969efec9b5f06a9bf49310d4390243d641'
EXPECTED_SIDECAR_BYTES = 1499866
EXPECTED_SIDECAR_SHA256 = '7e0dc5b0144faa5f0be407224eaf8070dd51199fa8a1d2f164b248fd6a06d5b2'
PART_SPECS = (
    ('m1_colab_train_val_backend_r1.tar.gz.part-00', 99614720, '53f30f08406fbef5b8451610b2fd44a1f2be42ca6d50f612b68e5ace285e66ff'),
    ('m1_colab_train_val_backend_r1.tar.gz.part-01', 99614720, '8451fe932760179d4b2e1a9cbdd21fd58744bc6b8f3d2a90e431cece7f546544'),
    ('m1_colab_train_val_backend_r1.tar.gz.part-02', 99614720, '867c935bf152ddfc3e7919547dccf3682291326b60b956a8100fc1a59b60e98d'),
    ('m1_colab_train_val_backend_r1.tar.gz.part-03', 99614720, '046fc8ae5d4e0a91196ccd6a6afa422fb1254baea7ae30bc3cab39602b534907'),
    ('m1_colab_train_val_backend_r1.tar.gz.part-04', 99614720, '6d3751d858232fada5d57e8e892f3e272cd5b9e34efb184971b3f4786842665d'),
    ('m1_colab_train_val_backend_r1.tar.gz.part-05', 99614720, '938106e4d34743e9b79109feb98d573f317afad44289f792f070f22e3924159b'),
    ('m1_colab_train_val_backend_r1.tar.gz.part-06', 99614720, '3497c1a10314c74d4b81ba9a5a49653b16ba2e2d52b424fd7655623547cb6408'),
    ('m1_colab_train_val_backend_r1.tar.gz.part-07', 68923760, '5f5ca85bc392310b2b5693ad9d0723f046eca9fd7980b45c944f22a3b3bc0eea'),
)
RUN_LABEL = 'smoke_seed_20261018_backend_r1'
STATE_DIR = DRIVE_ROOT / 'runs' / RUN_LABEL
REPO_DIR = Path('/content/Traffic_Sign_Detector')
REPO_URL = 'https://github.com/ScottPhantom/Traffic_SIgn_Detection.git'
SMOKE = True
SEED = 20261018
EPOCHS = 1 if SMOKE else 100
IMGSZ = 1280
BATCH = 8
DEVICE = '0'

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

assert SIDECAR.is_file(), f'Missing approved bundle sidecar: {SIDECAR}'
assert SIDECAR.stat().st_size == EXPECTED_SIDECAR_BYTES
assert sha256_file(SIDECAR) == EXPECTED_SIDECAR_SHA256
if DRIVE_BUNDLE.is_file():
    BUNDLE = DRIVE_BUNDLE
    bundle_source = 'single_archive'
else:
    parts = []
    for name, expected_bytes, expected_sha256 in PART_SPECS:
        part = DRIVE_ROOT / name
        assert part.is_file(), f'Missing approved train/val bundle part: {part}'
        assert part.stat().st_size == expected_bytes, part
        assert sha256_file(part) == expected_sha256, part
        parts.append(part)
    BUNDLE = Path('/content/m1_colab_train_val_backend_r1.tar.gz')
    if not BUNDLE.exists():
        with BUNDLE.open('xb') as destination:
            for part in parts:
                with part.open('rb') as source:
                    for chunk in iter(lambda: source.read(1024 * 1024), b''):
                        destination.write(chunk)
    bundle_source = 'verified_drive_parts'
assert BUNDLE.stat().st_size == EXPECTED_BUNDLE_BYTES
assert sha256_file(BUNDLE) == EXPECTED_BUNDLE_SHA256
STATE_DIR.mkdir(parents=True, exist_ok=True)
print({'mounted_bundle_preflight': 'PASS', 'bundle_source': bundle_source, 'bundle': str(BUNDLE), 'state_dir': str(STATE_DIR), 'epochs': EPOCHS})

In [ ]:
import json
import shutil
import subprocess
import sys
import tarfile

# The archive itself pins the exact repository commit that must consume it.
with tarfile.open(BUNDLE, 'r:gz') as archive:
    handle = archive.extractfile('m1_colab_bundle_manifest.json')
    assert handle is not None
    bundle_manifest = json.load(handle)
required_commit = bundle_manifest['repo_git_commit']
assert bundle_manifest['test_included'] is False
assert bundle_manifest['allowed_splits'] == ['train', 'val']

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', required_commit], cwd=REPO_DIR, check=True)
observed = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, check=True, capture_output=True, text=True
).stdout.strip()
assert observed == required_commit
print({'required_commit': required_commit, 'test_included': False})

In [ ]:
# Install the package manager only when the Colab image does not already provide it.
if shutil.which('uv') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
subprocess.run(['uv', '--version'], check=True)

In [ ]:
command = [
    sys.executable,
    str(REPO_DIR / 'detector/colab/run_d1_colab.py'),
    '--repo-root', str(REPO_DIR),
    '--bundle', str(BUNDLE),
    '--state-dir', str(STATE_DIR),
    '--seed', str(SEED),
    '--epochs', str(EPOCHS),
    '--imgsz', str(IMGSZ),
    '--batch', str(BATCH),
    '--device', DEVICE,
    '--tag', 'colab_smoke' if SMOKE else 'colab_final',
    '--download-approved-weight',
]
print('Running:', ' '.join(command))
subprocess.run(command, check=True)

In [ ]:
# Read back the persisted terminal outcome and packaged result from Drive.
execution = json.loads((STATE_DIR / 'colab_execution.json').read_text())
result_archives = sorted(STATE_DIR.glob('*_result.tar.gz'))
assert execution['status'] == 'SUCCEEDED', execution
assert execution['sealed_test_accessed'] is False
assert len(result_archives) == 1, result_archives
print(json.dumps(execution, indent=2, ensure_ascii=False))
print({'result_archive': str(result_archives[0]), 'bytes': result_archives[0].stat().st_size})